In [1]:
!pip install -q timm albumentations

In [2]:
import os, random, glob, warnings
import numpy as np
import pandas as pd
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torch.cuda.amp import GradScaler, autocast

import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, classification_report

warnings.filterwarnings("ignore")

In [3]:
SEED = 42
NUM_CLASSES = 13
IMG_SIZE = 448
NUM_WORKERS = 2
N_FOLDS = 5
LR_HEAD = 3e-4
LR_BACKBONE = 2e-5
WEIGHT_DECAY = 1e-4
EPOCHS_STAGE1 = 6
EPOCHS_STAGE2 = 12
WARMUP_EPOCHS = 2
MIXUP_ALPHA = 0.3
CUTMIX_ALPHA = 1.0
LABEL_SMOOTHING = 0.05
N_TTA = 8

BASE = "/kaggle/input/competitions/soda-challenge/soda_dataset"
TRAIN_DIR = os.path.join(BASE, "images", "train")
TEST_DIR = os.path.join(BASE, "images", "test")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

GPU_GB = 0
if torch.cuda.is_available():
    GPU_GB = torch.cuda.get_device_properties(0).total_memory / (1024 ** 3)

if GPU_GB >= 35:
    BATCH_SIZE = 8
    MODEL_CONFIGS = [
        "eva02_large_patch14_448.mim_m38m_ft_in22k_in1k",
        "eva02_base_patch14_448.mim_in22k_ft_in22k_in1k",
        "convnext_base.fb_in22k_ft_in1k",
    ]
else:
    BATCH_SIZE = 4
    MODEL_CONFIGS = [
        "eva02_base_patch14_448.mim_in22k_ft_in22k_in1k",
        "convnext_base.fb_in22k_ft_in1k",
    ]

MODEL_NAME = MODEL_CONFIGS[0]

print(f"Device: {DEVICE}")
print(f"GPU memory: {GPU_GB:.1f} GB")
print(f"Batch size: {BATCH_SIZE}")
print(f"Models: {MODEL_CONFIGS}")
print(f"Data root: {BASE}")

Device: cuda
GPU memory: 14.6 GB
Batch size: 4
Models: ['eva02_base_patch14_448.mim_in22k_ft_in22k_in1k', 'convnext_base.fb_in22k_ft_in1k']
Data root: /kaggle/input/competitions/soda-challenge/soda_dataset


In [4]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

seed_everything(SEED)

In [5]:
train_df = pd.read_csv(os.path.join(BASE, "train.csv"))
test_df = pd.read_csv(os.path.join(BASE, "test.csv"))

train_df["label_idx"] = train_df["label"] - 1

def find_image(img_id, directory):
    for ext in [".jpg", ".png", ".jpeg"]:
        p = os.path.join(directory, f"{img_id}{ext}")
        if os.path.exists(p):
            return p
    return os.path.join(directory, f"{img_id}.jpg")

train_df["path"] = train_df["id"].apply(lambda x: find_image(x, TRAIN_DIR))
test_df["path"] = test_df["id"].apply(lambda x: find_image(x, TEST_DIR))

print(f"Train: {len(train_df)}, Test: {len(test_df)}")
print(f"Classes: {train_df['label'].nunique()} (labels {train_df['label'].min()}-{train_df['label'].max()})")
print(train_df["label"].value_counts().sort_index())

Train: 1829, Test: 784
Classes: 13 (labels 1-13)
label
1     141
2     141
3     141
4     141
5     141
6     140
7     141
8     140
9     141
10    140
11    141
12    140
13    141
Name: count, dtype: int64


In [6]:
def get_train_transforms():
    return A.Compose([
        A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.7, 1.0), ratio=(0.75, 1.33)),
        A.HorizontalFlip(p=0.5),
        A.ShiftScaleRotate(shift_limit=0.1, scale_limit=0.15, rotate_limit=30,
                           border_mode=0, p=0.7),
        A.OneOf([
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.05),
            A.HueSaturationValue(hue_shift_limit=10, sat_shift_limit=20, val_shift_limit=20),
        ], p=0.6),
        A.OneOf([
            A.GaussNoise(std_range=(0.02, 0.08)),
            A.GaussianBlur(blur_limit=(3, 5)),
            A.MotionBlur(blur_limit=3),
        ], p=0.3),
        A.CoarseDropout(
            num_holes_range=(1, 4),
            hole_height_range=(int(IMG_SIZE * 0.05), int(IMG_SIZE * 0.1)),
            hole_width_range=(int(IMG_SIZE * 0.05), int(IMG_SIZE * 0.1)),
            fill=255, p=0.3),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_valid_transforms():
    return A.Compose([
        A.Resize(height=IMG_SIZE, width=IMG_SIZE),
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])

def get_tta_transforms(idx):
    base = [A.Resize(height=IMG_SIZE, width=IMG_SIZE)]
    if idx == 0:
        pass
    elif idx == 1:
        base.append(A.HorizontalFlip(p=1.0))
    elif idx == 2:
        base.append(A.ShiftScaleRotate(shift_limit=0, scale_limit=0, rotate_limit=10,
                                        border_mode=0, p=1.0))
    elif idx == 3:
        base.append(A.ShiftScaleRotate(shift_limit=0, scale_limit=0, rotate_limit=-10,
                                        border_mode=0, p=1.0))
    elif idx == 4:
        base.append(A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.9, 1.0), p=1.0))
    elif idx == 5:
        base.extend([A.HorizontalFlip(p=1.0),
                      A.ShiftScaleRotate(shift_limit=0, scale_limit=0, rotate_limit=10,
                                          border_mode=0, p=1.0)])
    elif idx == 6:
        base.append(A.ColorJitter(brightness=0.1, contrast=0.1, saturation=0.1, hue=0.02, p=1.0))
    elif idx == 7:
        base.append(A.RandomResizedCrop(size=(IMG_SIZE, IMG_SIZE), scale=(0.85, 0.95), p=1.0))
    base.extend([
        A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
        ToTensorV2(),
    ])
    return A.Compose(base)

In [7]:
class SODADataset(Dataset):
    def __init__(self, df, transform=None, is_test=False):
        self.df = df.reset_index(drop=True)
        self.transform = transform
        self.is_test = is_test

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img = np.array(Image.open(row["path"]).convert("RGB"))
        if self.transform:
            img = self.transform(image=img)["image"]
        if self.is_test:
            return img
        return img, row["label_idx"]

In [8]:
def mixup_data(x, y, alpha=0.3):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size = x.size(0)
    index = torch.randperm(batch_size, device=x.device)
    mixed_x = lam * x + (1 - lam) * x[index]
    y_a, y_b = y, y[index]
    return mixed_x, y_a, y_b, lam

def cutmix_data(x, y, alpha=1.0):
    lam = np.random.beta(alpha, alpha) if alpha > 0 else 1.0
    batch_size, _, H, W = x.size()
    index = torch.randperm(batch_size, device=x.device)
    cut_rat = np.sqrt(1.0 - lam)
    cut_w = int(W * cut_rat)
    cut_h = int(H * cut_rat)
    cx = np.random.randint(W)
    cy = np.random.randint(H)
    x1 = np.clip(cx - cut_w // 2, 0, W)
    y1 = np.clip(cy - cut_h // 2, 0, H)
    x2 = np.clip(cx + cut_w // 2, 0, W)
    y2 = np.clip(cy + cut_h // 2, 0, H)
    x[:, :, y1:y2, x1:x2] = x[index, :, y1:y2, x1:x2]
    lam = 1 - ((x2 - x1) * (y2 - y1) / (W * H))
    y_a, y_b = y, y[index]
    return x, y_a, y_b, lam

def mix_criterion(criterion, pred, y_a, y_b, lam):
    return lam * criterion(pred, y_a) + (1 - lam) * criterion(pred, y_b)

In [9]:
class SODAModel(nn.Module):
    def __init__(self, model_name, num_classes, pretrained=True):
        super().__init__()
        self.backbone = timm.create_model(model_name, pretrained=pretrained, num_classes=0)
        feat_dim = self.backbone.num_features
        self.head = nn.Sequential(
            nn.BatchNorm1d(feat_dim),
            nn.Dropout(0.4),
            nn.Linear(feat_dim, 512),
            nn.GELU(),
            nn.Dropout(0.3),
            nn.Linear(512, num_classes),
        )

    def forward(self, x):
        features = self.backbone(x)
        return self.head(features)

    def freeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = False

    def unfreeze_backbone(self):
        for param in self.backbone.parameters():
            param.requires_grad = True

In [10]:
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, label_smoothing=0.05):
        super().__init__()
        self.gamma = gamma
        self.label_smoothing = label_smoothing

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction="none",
                             label_smoothing=self.label_smoothing)
        pt = torch.exp(-ce)
        focal = ((1 - pt) ** self.gamma) * ce
        return focal.mean()

class CosineWarmupScheduler:
    def __init__(self, optimizer, warmup_steps, total_steps, min_lr=1e-7):
        self.optimizer = optimizer
        self.warmup_steps = warmup_steps
        self.total_steps = total_steps
        self.min_lr = min_lr
        self.base_lrs = [pg["lr"] for pg in optimizer.param_groups]
        self.step_count = 0

    def step(self):
        self.step_count += 1
        if self.step_count <= self.warmup_steps:
            scale = self.step_count / max(1, self.warmup_steps)
        else:
            progress = (self.step_count - self.warmup_steps) / max(
                1, self.total_steps - self.warmup_steps)
            scale = 0.5 * (1 + np.cos(np.pi * progress))
        for pg, base_lr in zip(self.optimizer.param_groups, self.base_lrs):
            pg["lr"] = max(self.min_lr, base_lr * scale)

    def get_lr(self):
        return [pg["lr"] for pg in self.optimizer.param_groups]

In [11]:
def train_one_epoch(model, loader, criterion, optimizer, scheduler, scaler, epoch):
    model.train()
    total_loss = 0
    all_preds, all_targets = [], []

    for batch_idx, (images, targets) in enumerate(loader):
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)

        use_mix = random.random() < 0.8
        if use_mix:
            if random.random() < 0.5:
                images, targets_a, targets_b, lam = mixup_data(images, targets, MIXUP_ALPHA)
            else:
                images, targets_a, targets_b, lam = cutmix_data(images, targets, CUTMIX_ALPHA)

        with autocast():
            logits = model(images)
            if use_mix:
                loss = mix_criterion(criterion, logits, targets_a, targets_b, lam)
            else:
                loss = criterion(logits, targets)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        scaler.step(optimizer)
        scaler.update()
        optimizer.zero_grad(set_to_none=True)
        scheduler.step()

        total_loss += loss.item()
        preds = logits.argmax(dim=1).detach().cpu()
        all_preds.extend(preds.numpy())
        all_targets.extend(targets.detach().cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_targets, all_preds, average="macro")
    return avg_loss, f1

@torch.no_grad()
def validate(model, loader, criterion):
    model.eval()
    total_loss = 0
    all_preds, all_probs = [], []
    all_targets = []

    for images, targets in loader:
        images = images.to(DEVICE, non_blocking=True)
        targets = targets.to(DEVICE, non_blocking=True)
        with autocast():
            logits = model(images)
            loss = criterion(logits, targets)
        total_loss += loss.item()
        probs = F.softmax(logits, dim=1).cpu().numpy()
        all_probs.append(probs)
        all_preds.extend(logits.argmax(dim=1).cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

    avg_loss = total_loss / len(loader)
    f1 = f1_score(all_targets, all_preds, average="macro")
    all_probs = np.concatenate(all_probs, axis=0)
    return avg_loss, f1, all_probs

@torch.no_grad()
def predict_tta(model, test_df, n_tta=N_TTA):
    model.eval()
    all_probs = np.zeros((len(test_df), NUM_CLASSES))
    for tta_idx in range(n_tta):
        transform = get_tta_transforms(tta_idx)
        ds = SODADataset(test_df, transform=transform, is_test=True)
        loader = DataLoader(ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=True)
        batch_probs = []
        for images in loader:
            images = images.to(DEVICE, non_blocking=True)
            with autocast():
                logits = model(images)
            batch_probs.append(F.softmax(logits, dim=1).cpu().numpy())
        all_probs += np.concatenate(batch_probs, axis=0)
    return all_probs / n_tta

In [12]:
def run_cv(model_name):
    """Train 5-fold CV for one backbone. Returns (oof_probs, test_probs, fold_scores)."""
    oof = np.zeros((len(train_df), NUM_CLASSES))
    test = np.zeros((len(test_df), NUM_CLASSES))
    skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
    scores = []

    for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label_idx"])):
        print(f"\n{'='*60}\n  {model_name}\n  FOLD {fold+1}/{N_FOLDS}\n{'='*60}")
        tr_df = train_df.iloc[train_idx]
        va_df = train_df.iloc[val_idx]

        train_ds = SODADataset(tr_df, transform=get_train_transforms())
        val_ds = SODADataset(va_df, transform=get_valid_transforms())
        train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                                  num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
        val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE * 2, shuffle=False,
                                num_workers=NUM_WORKERS, pin_memory=True)

        model = SODAModel(model_name, NUM_CLASSES, pretrained=True).to(DEVICE)
        criterion = FocalLoss(gamma=2.0, label_smoothing=LABEL_SMOOTHING)
        scaler = GradScaler()
        ckpt = f"best_{model_name.split('.')[0]}_fold{fold}.pt"

        print("\nStage 1: Training head only...")
        model.freeze_backbone()
        optimizer = torch.optim.AdamW(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=LR_HEAD, weight_decay=WEIGHT_DECAY)
        steps_per_epoch = len(train_loader)
        scheduler = CosineWarmupScheduler(optimizer, warmup_steps=steps_per_epoch,
                                          total_steps=EPOCHS_STAGE1 * steps_per_epoch)
        best_f1 = 0
        for epoch in range(EPOCHS_STAGE1):
            tl, tf = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, epoch)
            vl, vf, _ = validate(model, val_loader, criterion)
            print(f"  Epoch {epoch+1}/{EPOCHS_STAGE1} | Train Loss: {tl:.4f} F1: {tf:.4f} | "
                  f"Val Loss: {vl:.4f} F1: {vf:.4f} | LR: {scheduler.get_lr()[0]:.2e}")
            if vf > best_f1:
                best_f1 = vf
                torch.save(model.state_dict(), ckpt)

        print(f"\nStage 2: Fine-tuning full model (best head F1: {best_f1:.4f})...")
        model.load_state_dict(torch.load(ckpt, weights_only=True))
        model.unfreeze_backbone()
        optimizer = torch.optim.AdamW([
            {"params": model.backbone.parameters(), "lr": LR_BACKBONE},
            {"params": model.head.parameters(), "lr": LR_BACKBONE * 5},
        ], weight_decay=WEIGHT_DECAY)
        scheduler = CosineWarmupScheduler(optimizer, warmup_steps=WARMUP_EPOCHS * steps_per_epoch,
                                          total_steps=EPOCHS_STAGE2 * steps_per_epoch)
        patience = 0
        for epoch in range(EPOCHS_STAGE2):
            tl, tf = train_one_epoch(model, train_loader, criterion, optimizer, scheduler, scaler, epoch)
            vl, vf, _ = validate(model, val_loader, criterion)
            print(f"  Epoch {epoch+1}/{EPOCHS_STAGE2} | Train Loss: {tl:.4f} F1: {tf:.4f} | "
                  f"Val Loss: {vl:.4f} F1: {vf:.4f} | LR: {scheduler.get_lr()[0]:.2e}")
            if vf > best_f1:
                best_f1 = vf
                torch.save(model.state_dict(), ckpt)
                patience = 0
            else:
                patience += 1
                if patience >= 5:
                    print(f"  Early stopping at epoch {epoch+1}")
                    break

        model.load_state_dict(torch.load(ckpt, weights_only=True))
        _, _, val_probs = validate(model, val_loader, criterion)
        oof[val_idx] = val_probs
        fold_f1 = f1_score(va_df["label_idx"], np.argmax(val_probs, axis=1), average="macro")
        scores.append(fold_f1)
        print(f"\n  Fold {fold+1} F1: {fold_f1:.5f}")

        print(f"  Running {N_TTA}x TTA on test set...")
        test += predict_tta(model, test_df, n_tta=N_TTA)

        del model, optimizer, scaler
        torch.cuda.empty_cache()

    test /= N_FOLDS
    return oof, test, scores

model_oof, model_test, model_scores = {}, {}, {}
for mname in MODEL_CONFIGS:
    oof, test, scores = run_cv(mname)
    model_oof[mname] = oof
    model_test[mname] = test
    model_scores[mname] = scores
    key = mname.split('.')[0]
    np.save(f"oof_{key}.npy", oof)
    np.save(f"test_{key}.npy", test)
    f1 = f1_score(train_df["label_idx"].values, np.argmax(oof, axis=1), average="macro")
    print(f"\n>>> {mname}\n    OOF macro F1: {f1:.5f} | folds {[f'{s:.4f}' for s in scores]}")


  eva02_base_patch14_448.mim_in22k_ft_in22k_in1k
  FOLD 1/5


model.safetensors:   0%|          | 0.00/348M [00:00<?, ?B/s]


Stage 1: Training head only...
  Epoch 1/6 | Train Loss: 2.1308 F1: 0.1172 | Val Loss: 1.8126 F1: 0.2688 | LR: 3.00e-04
  Epoch 2/6 | Train Loss: 1.8246 F1: 0.2379 | Val Loss: 1.5147 F1: 0.3558 | LR: 2.71e-04
  Epoch 3/6 | Train Loss: 1.7215 F1: 0.2725 | Val Loss: 1.3682 F1: 0.4339 | LR: 1.96e-04
  Epoch 4/6 | Train Loss: 1.6170 F1: 0.3024 | Val Loss: 1.2657 F1: 0.4636 | LR: 1.04e-04
  Epoch 5/6 | Train Loss: 1.5513 F1: 0.3342 | Val Loss: 1.2774 F1: 0.4875 | LR: 2.86e-05
  Epoch 6/6 | Train Loss: 1.5037 F1: 0.3477 | Val Loss: 1.2535 F1: 0.4864 | LR: 1.00e-07

Stage 2: Fine-tuning full model (best head F1: 0.4875)...
  Epoch 1/12 | Train Loss: 1.5369 F1: 0.3548 | Val Loss: 1.0589 F1: 0.5341 | LR: 1.00e-05
  Epoch 2/12 | Train Loss: 1.4512 F1: 0.3623 | Val Loss: 1.1679 F1: 0.4935 | LR: 2.00e-05
  Epoch 3/12 | Train Loss: 1.4852 F1: 0.3594 | Val Loss: 0.8447 F1: 0.6386 | LR: 1.95e-05
  Epoch 4/12 | Train Loss: 1.2648 F1: 0.4488 | Val Loss: 0.7746 F1: 0.6115 | LR: 1.81e-05
  Epoch 5/12 | 

model.safetensors:   0%|          | 0.00/354M [00:00<?, ?B/s]


Stage 1: Training head only...
  Epoch 1/6 | Train Loss: 2.0735 F1: 0.1517 | Val Loss: 1.5754 F1: 0.3705 | LR: 3.00e-04
  Epoch 2/6 | Train Loss: 1.7149 F1: 0.2684 | Val Loss: 1.2586 F1: 0.5069 | LR: 2.71e-04
  Epoch 3/6 | Train Loss: 1.6051 F1: 0.3285 | Val Loss: 1.1603 F1: 0.5084 | LR: 1.96e-04
  Epoch 4/6 | Train Loss: 1.4686 F1: 0.3596 | Val Loss: 1.1259 F1: 0.5035 | LR: 1.04e-04
  Epoch 5/6 | Train Loss: 1.4593 F1: 0.3926 | Val Loss: 1.0541 F1: 0.5555 | LR: 2.86e-05
  Epoch 6/6 | Train Loss: 1.3617 F1: 0.4031 | Val Loss: 1.0660 F1: 0.5278 | LR: 1.00e-07

Stage 2: Fine-tuning full model (best head F1: 0.5555)...
  Epoch 1/12 | Train Loss: 1.3618 F1: 0.3955 | Val Loss: 0.9997 F1: 0.5640 | LR: 1.00e-05
  Epoch 2/12 | Train Loss: 1.2831 F1: 0.4426 | Val Loss: 0.8413 F1: 0.6298 | LR: 2.00e-05
  Epoch 3/12 | Train Loss: 1.2040 F1: 0.4453 | Val Loss: 0.8485 F1: 0.6264 | LR: 1.95e-05
  Epoch 4/12 | Train Loss: 1.1127 F1: 0.4880 | Val Loss: 0.6791 F1: 0.7162 | LR: 1.81e-05
  Epoch 5/12 | 

In [13]:
true_labels = train_df["label_idx"].values
print(f"{'model':55s} OOF F1")
print("-" * 70)
for mname in MODEL_CONFIGS:
    f1 = f1_score(true_labels, np.argmax(model_oof[mname], axis=1), average="macro")
    print(f"{mname:55s} {f1:.5f}")

model                                                   OOF F1
----------------------------------------------------------------------
eva02_base_patch14_448.mim_in22k_ft_in22k_in1k          0.82944
convnext_base.fb_in22k_ft_in1k                          0.80427


In [14]:
true_labels = train_df["label_idx"].values
names = list(MODEL_CONFIGS)
oofs = [model_oof[n] for n in names]
tests = [model_test[n] for n in names]

def f1_of(prob):
    return f1_score(true_labels, np.argmax(prob, axis=1), average="macro")

single = [f1_of(o) for o in oofs]
for n, s in zip(names, single):
    print(f"  single {n.split('.')[0]:40s} {s:.5f}")

equal_oof = np.mean(oofs, axis=0)
print(f"\nEqual-weight ensemble OOF F1: {f1_of(equal_oof):.5f}")

ens = [int(np.argmax(single))]
weights = [0] * len(names)
weights[ens[0]] += 1
for _ in range(30):
    best_s, best_i = -1, -1
    for i in range(len(names)):
        cand = np.mean([oofs[j] for j in ens + [i]], axis=0)
        s = f1_of(cand)
        if s > best_s:
            best_s, best_i = s, i
    ens.append(best_i)
    weights[best_i] += 1

w = np.array(weights, dtype=float)
w /= w.sum()
greedy_oof = np.tensordot(w, np.array(oofs), axes=([0], [0]))
greedy_test = np.tensordot(w, np.array(tests), axes=([0], [0]))
print("Greedy weights:", {names[i].split('.')[0]: round(float(w[i]), 3) for i in range(len(names))})
print(f"Greedy ensemble OOF F1: {f1_of(greedy_oof):.5f}")

if f1_of(equal_oof) >= f1_of(greedy_oof):
    ENSEMBLE_OOF, ENSEMBLE_TEST = equal_oof, np.mean(tests, axis=0)
    print("\n=> Using EQUAL-weight ensemble")
else:
    ENSEMBLE_OOF, ENSEMBLE_TEST = greedy_oof, greedy_test
    print("\n=> Using GREEDY-weighted ensemble")

print(f"\n{'='*60}")
print(f"  FINAL ENSEMBLE OOF MACRO F1: {f1_of(ENSEMBLE_OOF):.5f}")
print(f"{'='*60}")
print(classification_report(
    true_labels, np.argmax(ENSEMBLE_OOF, axis=1),
    labels=list(range(NUM_CLASSES)),
    target_names=[f"Class_{i}" for i in range(1, NUM_CLASSES + 1)]
))

  single eva02_base_patch14_448                   0.82944
  single convnext_base                            0.80427

Equal-weight ensemble OOF F1: 0.83693
Greedy weights: {'eva02_base_patch14_448': 0.452, 'convnext_base': 0.548}
Greedy ensemble OOF F1: 0.84024

=> Using GREEDY-weighted ensemble

  FINAL ENSEMBLE OOF MACRO F1: 0.84024
              precision    recall  f1-score   support

     Class_1       0.96      0.94      0.95       141
     Class_2       0.85      0.79      0.82       141
     Class_3       0.78      0.75      0.77       141
     Class_4       0.80      0.73      0.77       141
     Class_5       0.67      0.87      0.75       141
     Class_6       0.96      0.94      0.95       140
     Class_7       0.78      0.76      0.77       141
     Class_8       0.85      0.71      0.77       140
     Class_9       0.91      0.90      0.90       141
    Class_10       0.79      0.79      0.79       140
    Class_11       0.87      0.94      0.90       141
    Class_12   

In [15]:
pred_labels = np.argmax(ENSEMBLE_TEST, axis=1) + 1
submission = pd.DataFrame({"id": test_df["id"], "label": pred_labels.astype(int)})
submission.to_csv("submission.csv", index=False)
print("Saved submission.csv")
print(submission.head(10))
print(f"\nLabel range: {submission['label'].min()}-{submission['label'].max()}")
print(submission["label"].value_counts().sort_index())

np.save("ensemble_oof.npy", ENSEMBLE_OOF)
np.save("ensemble_test.npy", ENSEMBLE_TEST)

Saved submission.csv
   id  label
0   1      7
1   2     10
2   3      2
3   4      1
4   5     11
5   6     12
6   7      6
7   8      6
8   9      5
9  10     10

Label range: 1-13
label
1     56
2     57
3     59
4     58
5     77
6     61
7     54
8     61
9     65
10    55
11    66
12    59
13    56
Name: count, dtype: int64
